# RSO-113: Compare thermal behavior across different louver configurations

Check whether different louver configurations lead to noticeable differences in how quickly or how strongly the dome and telescope temperatures respond to outside conditions.



Expected results:

**Plots**: temperature vs time with configuration highlighted

**Comparison tables**: how much temperatures change for each configuration

**Variation view**: results grouped by similar wind conditions (to avoid mixing very different nights)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time
import matplotlib.dates as mdates
import matplotlib.cm as cm
import seaborn as sns

from lsst.summit.utils.efdUtils import getEfdData, makeEfdClient
from lsst.summit.utils.tmaUtils import TMAEventMaker, TMAState

In [ ]:
t_start_period = Time("2026-01-31T00:00:00Z", scale="utc")
t_end_period = Time("2026-02-15T00:00:00Z", scale="utc")

efd_client = makeEfdClient()

# Queries

In [ ]:
def query_setlouvers(start, end):
    df_louvers = getEfdData(
        client=efd_client,
        topic="lsst.sal.MTDome.command_setLouvers",
        columns=["*"],
        begin=start,
        end=end,
    )

    return df_louvers

In [ ]:
def query_essTemperature(start, end):
    df_esstemperature = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.temperature",
        columns=['location', 'private_identity','sensorName', 'temperatureItem0','timestamp'],
        begin=start,
        end=end,
    )

    return df_esstemperature

In [ ]:
def query_airFlow(start, end):
    df_airFlow = getEfdData(
        client=efd_client,
        topic="lsst.sal.ESS.airFlow",
        columns=['location', 'private_identity','sensorName', 'direction','timestamp', 'directionStdDev', 'speed', 'speedStdDev'],
        begin=start,
        end=end,
    )

    return df_airFlow

# Plot functions

# Configuration of louvers

In [ ]:
df_setlouvers = query_setlouvers(t_start_period, t_end_period)

In [ ]:
# Copy current index into a new column before any merge
df_setlouvers['time_stamp'] = df_setlouvers.index

In [ ]:
# Select all columns that start with "position"
position_cols = df_setlouvers.filter(regex=r'^position').columns

# Sort columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position', '')))

# Compute unique combinations
combination_counts = (
    df_setlouvers[position_cols]
    .value_counts()
    .reset_index(name='count')
)

# Create configuration ID (1 to N)
combination_counts['louvers_conf'] = range(1, len(combination_counts) + 1)

# Merge configuration ID back into original dataframe
df_setlouvers = df_setlouvers.merge(
    combination_counts[position_cols + ['louvers_conf']],
    on=position_cols,
    how='left'
)

print(f"Number of unique configurations detected: {len(combination_counts)}\n")

# Print configurations showing only non-zero positions
for _, row in combination_counts.iterrows():
    
    conf_id = row['louvers_conf']
    count = row['count']
    
    print(f"Configuration {conf_id} (appears {count} times):")
    
    # Extract position values
    config = row[position_cols]
    
    # Keep only non-zero values
    non_zero = config[config != 0]
    
    if len(non_zero) == 0:
        print("  All positions are 0")
    else:
        for col, val in non_zero.items():
            print(f"  {col}: {val}")
    
    print("-" * 40)

# Prepare the data

In [ ]:
df_esstemperature = query_essTemperature(t_start_period, t_end_period)

"ESS:111": "Temperature dome inside",
"ESS:301": "Temperature ess weather station",
"ESS:113": "Temperature m1m3 inside",
"ESS:112": "Temperature m2 inside"
    

In [ ]:
# Keep only the ESS identities with 111 and 301
df_temp = df_esstemperature[
    df_esstemperature['private_identity'].isin(['ESS:111', 'ESS:112', 'ESS:113', 'ESS:301'])
].copy()

In [ ]:
# Add in df_setlouvers configuration the end of each configuration
df_setlouvers['time_end'] = df_setlouvers['time_stamp'].shift(-1)
df_temp['time'] = pd.to_datetime(df_temp['timestamp'], unit='s')
max_time = df_temp['time'].max()
df_setlouvers['time_end'] = df_setlouvers['time_end'].fillna(max_time)

In [ ]:
# Column positions
position_cols = df_setlouvers.filter(regex=r'^position').columns
position_cols = sorted(position_cols, key=lambda x: int(x.replace('position','')))

# Function to labels
def build_conf_label(row):
    config = row[position_cols]
    non_zero = config[config != 0]

    if len(non_zero) == 0:
        return f"Configuration {row['louvers_conf']}: all closed"

    lines = [f"Configuration {row['louvers_conf']}:"]
    for col, val in non_zero.items():
        lines.append(f"{col}: {int(val)}")
    return "\n".join(lines)

df_setlouvers['conf_label'] = df_setlouvers.apply(build_conf_label, axis=1)

In [ ]:
# all dates in the same format
df_temp["time"] = pd.to_datetime(df_temp["time"], utc=True)

df_setlouvers["time_stamp"] = pd.to_datetime(df_setlouvers["time_stamp"], utc=True)
df_setlouvers["time_end"] = pd.to_datetime(df_setlouvers["time_end"], utc=True)

In [ ]:
# Sort times
df_temp = df_temp.sort_values("time")
df_setlouvers = df_setlouvers.sort_values("time_stamp")

In [ ]:
df_temp_merge = pd.merge_asof(
    df_temp,
    df_setlouvers,
    left_on="time",
    right_on="time_stamp",
    direction="backward"
)

In [ ]:
df_temp_merge = df_temp_merge[
    df_temp_merge["time"] <= df_temp_merge["time_end"]
]

In [ ]:
df_temp_merge.head()

# temperature vs time with configuration highlighted

Since I didn't know which one was the “highlighted configuration” I set it up so that I could view the graphs with all

In [ ]:
color_map = {
    "ESS:111": "blue",
    "ESS:112": "orange",
    "ESS:113": "green",
    "ESS:301": "red"
}

for _, row in df_setlouvers.iterrows():
    conf = row["conf_label"]
    t0 = row["time_stamp"]
    t1 = row["time_end"]
    
    df_interval = df_temp_merge[
        (df_temp_merge["time"] >= t0) &
        (df_temp_merge["time"] <= t1)
    ]
    
    if df_interval.empty:
        continueshow()
    
    plt.figure()
    
    for sensor in df_interval["private_identity_x"].unique():
        df_s = df_interval[df_interval["private_identity_x"] == sensor]
        plt.plot(
           df_s["time"],
           df_s["temperatureItem0"],
           label=sensor,
           color=color_map.get(sensor, "black")
        )
    
    plt.title(f"{conf} \n {t0} → {t1}")
    plt.xlabel("Time")
    plt.ylabel("Temperature")
    plt.legend()
    plt.show()

# Comparison tables: how much temperatures change for each configuration

### calculate temperature change by interval

In [ ]:
results = []

for _, row in df_setlouvers.iterrows():
    conf_num = row["louvers_conf"]
    conf = row["conf_label"]
    t0 = row["time_stamp"]
    t1 = row["time_end"]
    
    df_interval = df_temp_merge[
        (df_temp_merge["time"] >= t0) &
        (df_temp_merge["time"] <= t1)
    ]
    
    if df_interval.empty:
        continue
    
    for sensor in df_interval["private_identity_x"].unique():
        df_s = df_interval[df_interval["private_identity_x"] == sensor]
        
        temp_change = df_s["temperatureItem0"].max() - df_s["temperatureItem0"].min()
        
        results.append({
            "louvers_conf": conf_num,
            "conf_label": conf,
            "sensor": sensor,
            "delta_T": temp_change,
            "duration_min": (t1 - t0).total_seconds() / 60
        })

In [ ]:
df_changes = pd.DataFrame(results)

In [ ]:
df_changes.head()

## Summary

In [ ]:
df_summary = (
    df_changes
    .groupby(["louvers_conf","conf_label", "sensor"])
    .agg(
        mean_delta_T=("delta_T", "mean"),
        median_delta_T=("delta_T", "median"),
        std_delta_T=("delta_T", "std"),
        n_intervals=("delta_T", "count")
    )
    .reset_index()
)

In [ ]:
df_summary

In [ ]:
color_map = {
    "ESS:111": "blue",
    "ESS:112": "orange",
    "ESS:113": "green",
    "ESS:301": "red"
}

plt.figure(figsize=(10,6))

for sensor in df_summary["sensor"].unique():
    df_s = df_summary[df_summary["sensor"] == sensor]
    
    plt.errorbar(
        df_s["louvers_conf"],
        df_s["mean_delta_T"],
        yerr=df_s["std_delta_T"],
        fmt='o',
        label=sensor,
        color=color_map.get(sensor, "black"),
        capsize=4
    )

plt.xlabel("Louvers Configuration (ID)")
plt.ylabel("Mean ΔT")
plt.title("Temperature Change per Louver Configuration")
plt.legend(title="Sensor")


conf_map = df_summary[["louvers_conf", "conf_label"]].drop_duplicates()

text = "\n".join(
    [f"{row.louvers_conf}: {row.conf_label}" for _, row in conf_map.iterrows()]
)

plt.figtext(1.0, -0.2, text, ha="center", fontsize=9)
plt.grid(alpha=0.3)



In [ ]:
df_duration = (
    df_changes
    .groupby("louvers_conf")["duration_min"]
    .sum()   # o .mean() 
    .reset_index()
)

In [ ]:
color_map = {
    "ESS:111": "blue",
    "ESS:112": "orange",
    "ESS:113": "green",
    "ESS:301": "red"
}

plt.figure(figsize=(10,6))

for sensor in df_summary["sensor"].unique():
    df_s = df_summary[df_summary["sensor"] == sensor]
    
    plt.errorbar(
        df_s["louvers_conf"],
        df_s["mean_delta_T"],
        yerr=df_s["std_delta_T"],
        fmt='o',
        label=sensor,
        color=color_map.get(sensor, "black"),
        capsize=4
    )

plt.xlabel("Louvers Configuration (ID)")
plt.ylabel("Mean ΔT")
plt.title("Temperature Change per Louver Configuration")
plt.legend(title="Sensor")


conf_map = df_summary[["louvers_conf", "conf_label"]].drop_duplicates()

text = "\n".join(
    [f"{row.louvers_conf}: {row.conf_label}" for _, row in conf_map.iterrows()]
)

conf_duration_map = dict(zip(df_duration["louvers_conf"], df_duration["duration_min"]))

xticks = sorted(df_summary["louvers_conf"].unique())

xticklabels = [
    f"{conf}\n({conf_duration_map.get(conf, 0):.0f} min)"
    for conf in xticks
]

#plt.xticks(xticks, xticklabels, rotation=90, ha='center')

plt.xticks(xticks, xticklabels, rotation=45, ha='right')
plt.subplots_adjust(bottom=0.25)

plt.figtext(1.0, -0.2, text, ha="center", fontsize=9)
plt.grid(alpha=0.3)


## Same plot but separate by configuration and time in each configuration

In [ ]:
results = []

for _, row in df_setlouvers.iterrows():
    conf_num = row["louvers_conf"]
    conf_label = row["conf_label"]
    t0 = row["time_stamp"]
    t1 = row["time_end"]
    
    df_interval = df_temp_merge[
        (df_temp_merge["time"] >= t0) &
        (df_temp_merge["time"] <= t1)
    ]
    
    if df_interval.empty:
        continue
    
    duration = (t1 - t0).total_seconds() / 60  # minutos
    
    for sensor in df_interval["private_identity_x"].unique():
        df_s = df_interval[df_interval["private_identity_x"] == sensor]
        
        results.append({
            "louvers_conf": conf_num,
            "conf_label": conf_label,
            "sensor": sensor,
            "time_stamp": t0,
            "time_end": t1,
            "mean_T": df_s["temperatureItem0"].mean(),
            "std_T": df_s["temperatureItem0"].std(),
            "duration_min": duration
          })

In [ ]:
df_intervals = pd.DataFrame(results)

In [ ]:

color_map = {
    "ESS:111": "blue",
    "ESS:112": "orange",
    "ESS:113": "green",
    "ESS:301": "red"
}

configs = sorted(df_intervals["louvers_conf"].unique())

for conf in configs:
    df_c = df_intervals[df_intervals["louvers_conf"] == conf]
    
    plt.figure(figsize=(8,5))
    
    for sensor in ["ESS:111", "ESS:112", "ESS:113", "ESS:301"]:
        df_s = df_c[df_c["sensor"] == sensor]
        
        if df_s.empty:
            continue
        
        plt.errorbar(
            df_s["duration_min"],
            df_s["mean_T"],
            yerr=df_s["std_T"],
            fmt='o',
            label=sensor,
            color=color_map.get(sensor, "black"),
            capsize=4
        )
        
        # 👉 añadir etiquetas SOLO para un sensor
        if sensor == "ESS:111":
            for _, row in df_s.iterrows():
                plt.annotate(
                    f'{row["duration_min"]:.0f}',
                    (row["duration_min"], row["mean_T"]),
                    textcoords="offset points",
                    xytext=(5,8),
                    fontsize=7
                )

    # título con descripción
    conf_label = df_c["conf_label"].iloc[0]
    
    plt.title(f"Config {conf}\n{conf_label}")
    plt.xlabel("Duration (min)")
    plt.ylabel("Mean Temperature")
    plt.legend(title="Sensor")
    plt.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Variation view: results grouped by similar wind conditions (to avoid mixing very different nights)



In [ ]:
df_airflow = query_airFlow(t_start_period, t_end_period)

In [ ]:
df_airflow

In [ ]:
# Convert format of time
df_airflow["time"] = pd.to_datetime(df_airflow["timestamp"], unit="s", utc=True)

## Assign a wind speed to each interval

In [ ]:
wind_results = []

for _, row in df_setlouvers.iterrows():
    t0 = row["time_stamp"]
    t1 = row["time_end"]
    
    df_wind = df_airflow[
        (df_airflow["time"] >= t0) &
        (df_airflow["time"] <= t1)
    ]
    
    if df_wind.empty:
        continue
    
    wind_results.append({
        "time_stamp": t0,
        "time_end": t1,
        "wind_speed_mean": df_wind["speed"].mean(),
        "wind_speed_std": df_wind["speedStdDev"].mean(),
        "wind_dir_mean": df_wind["direction"].mean()
    })

In [ ]:
df_wind_intervals = pd.DataFrame(wind_results)

In [ ]:
df_wind_intervals.head()

## Combine with your temperature ranges

In [ ]:
df_full = pd.merge(
    df_intervals,
    df_wind_intervals,
    on=["time_stamp", "time_end"],
    how="inner"
)

In [ ]:
df_full

## group by wind conditions

In [ ]:
df_full["wind_bin"] = pd.cut(
    df_full["wind_speed_mean"],
    bins=[0, 2, 5, 10, 20],
    labels=["low", "medium", "high", "very high"]
)

In [ ]:
df_full["wind_dir_bin"] = pd.cut(
    df_full["wind_dir_mean"],
    bins=[0, 90, 180, 270, 360],
    labels=["N-E", "E-S", "S-W", "W-N"]
)

In [ ]:
df_grouped = (
    df_full
    .groupby(["wind_bin", "louvers_conf", "sensor"])
    .agg(
        mean_T=("mean_T", "mean"),
        std_T=("mean_T", "std"),
        count=("mean_T", "count")
    )
    .reset_index()
)

In [ ]:
df_plot = (
    df_full
    .groupby(["wind_bin", "wind_dir_bin", "louvers_conf", "sensor"])
    .agg(
        mean_T=("mean_T", "mean"),
        std_T=("mean_T", "std"),
        n=("mean_T", "count")
    )
    .reset_index()
)

In [ ]:
color_map = {
    "ESS:111": "blue",
    "ESS:112": "orange",
    "ESS:113": "green",
    "ESS:301": "red"
}

wind_bins = df_plot["wind_bin"].dropna().unique()

for wb in wind_bins:
    df_w = df_plot[df_plot["wind_bin"] == wb]
    
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    axes = axes.flatten()
    
    for i, wd in enumerate(df_w["wind_dir_bin"].dropna().unique()):
        ax = axes[i]
        df_wd = df_w[df_w["wind_dir_bin"] == wd]
        
        for sensor in df_wd["sensor"].unique():
            df_s = df_wd[df_wd["sensor"] == sensor]
            
            ax.errorbar(
                df_s["louvers_conf"],
                df_s["mean_T"],
                yerr=df_s["std_T"],
                fmt='o',
                label=sensor,
                color=color_map.get(sensor, "black"),
                capsize=4
            )
        
        ax.set_title(f"Dir: {wd}")
        ax.set_xlabel("Config")
        ax.set_ylabel("Temp")
        ax.set_xlim(0, 16)   # 👈 AQUÍ
        ax.grid(alpha=0.3)
    
    fig.suptitle(f"Wind speed: {wb}")
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper right")
    
    plt.tight_layout()
    plt.show()